# 08 — Full-Context Linear-Attention DT

Same model as `05_train_linearattn_dt.ipynb` (`LinearDecisionTransformer`, `phi(x) = elu(x) + 1` causal linear attention), but each training example is **one full session** instead of a sliding 32-step window.

Motivation: the suffix-sum RTG signal references every reward to the end of the session — across trial boundaries — but a 32-step window almost never contains the trial boundary or the rewards the RTG is pointing at. Linear attention's `O(L)` cost makes the full session affordable; this notebook tests whether a single-stream model with full context already closes the cross-trial gap (the cheaper half of the ablation pair, paired with notebook 09's hybrid).

Right-pad each session to `FULL_CTX`; mask loss at padded positions. Sessions longer than `FULL_CTX` are truncated to their tail (the terminal reward step is the most informative for RTG semantics).


## 1. Setup


In [ ]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import sys; sys.path.insert(0, '../scripts')
import train_dt as T  # reuses encode_session for per-step encoding

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.linear_decision_transformer import (
    LinearDTConfig, LinearDecisionTransformer,
)

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb08'); RUN_DIR.mkdir(parents=True, exist_ok=True)

FULL_CTX     = 4096     # covers ~99% of sessions; longer sessions truncated to tail
EMBED_DIM    = 60       # matched to GridCellEncoder.output_dim
NUM_HEADS    = 4        # d_head = 60/4 = 15
NUM_LAYERS   = 2
POS_ENCODING = 'learned'
LR           = 5e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 4        # full sessions are big — batches are small
EPOCHS       = 30
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None     # None = all; set e.g. 50 for a smoke run

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  d_head={EMBED_DIM // NUM_HEADS}  FULL_CTX={FULL_CTX}')


## 2. Load and pack each session as one example


In [ ]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)

encoder = GridCellEncoder()
assert encoder.output_dim == EMBED_DIM

def pack_session(sdf, full_ctx):
    arrays = T.encode_session(sdf, encoder)
    n = arrays['state'].shape[0]
    if n > full_ctx:
        arrays = {k: v[-full_ctx:] for k, v in arrays.items()}
        n = full_ctx
    state  = np.zeros((full_ctx, EMBED_DIM), dtype=np.float32); state[:n]  = arrays['state']
    action = np.zeros((full_ctx, 5),         dtype=np.float32); action[:n] = arrays['action']
    rtg    = np.zeros((full_ctx, 1),         dtype=np.float32); rtg[:n]    = arrays['rtg']
    mask   = np.zeros((full_ctx,),           dtype=np.float32); mask[:n]   = 1.0
    return state, action, rtg, mask

sids = df['session_id'].unique().tolist()
states  = np.empty((len(sids), FULL_CTX, EMBED_DIM), dtype=np.float32)
actions = np.empty((len(sids), FULL_CTX, 5),         dtype=np.float32)
rtgs    = np.empty((len(sids), FULL_CTX, 1),         dtype=np.float32)
masks   = np.empty((len(sids), FULL_CTX),            dtype=np.float32)
for i, sid in enumerate(sids):
    sdf = df[df['session_id'] == sid].sort_values('step')
    states[i], actions[i], rtgs[i], masks[i] = pack_session(sdf, FULL_CTX)

valid_counts = masks.sum(axis=1).astype(int)
print(f'sessions packed: {len(sids)}   valid-len mean={valid_counts.mean():.0f} '
      f'min={valid_counts.min()}  max={valid_counts.max()}')

full = TensorDataset(
    torch.from_numpy(rtgs), torch.from_numpy(states),
    torch.from_numpy(actions), torch.from_numpy(masks),
)
n = len(full); n_val = max(1, int(n * VAL_FRAC)); n_train = n - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f'sessions: train={n_train}  val={n_val}')


## 3. Train (loss masked on padding)


In [ ]:
cfg = LinearDTConfig(
    embed_dim=EMBED_DIM, num_actions=5, context_size=FULL_CTX,
    num_heads=NUM_HEADS, num_layers=NUM_LAYERS, pos_encoding=POS_ENCODING,
)
model = LinearDecisionTransformer(cfg).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce = nn.CrossEntropyLoss(reduction='none')
n_params = sum(p.numel() for p in model.parameters())
print(f'params={n_params:,}')

def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    sum_loss = sum_correct = sum_tokens = 0.0
    ctx_mgr = torch.enable_grad() if train else torch.no_grad()
    with ctx_mgr:
        for rtg, state, action, mask in loader:
            rtg    = rtg.to(device)
            state  = state.to(device)
            action = action.to(device)
            mask   = mask.to(device)
            logits = model(rtg, state, action)
            targets = action.argmax(dim=-1)
            losses = ce(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
            losses = losses.reshape(targets.shape)
            denom = mask.sum().clamp_min(1.0)
            loss = (losses * mask).sum() / denom
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            sum_loss    += (losses * mask).sum().item()
            sum_correct += ((logits.argmax(-1) == targets).float() * mask).sum().item()
            sum_tokens  += mask.sum().item()
    return sum_loss / max(sum_tokens, 1), sum_correct / max(sum_tokens, 1)

history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  ({time.time()-t0:.0f}s)')

ckpt = RUN_DIR / 'model.pt'
torch.save({'state_dict': model.state_dict(), 'cfg': cfg.__dict__,
            'arch': 'linear_fullctx'}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')


## 4. Inference movie

Grow a per-step (rtg, state, action) buffer over the rollout — the model conditions on the entire history each step. Truncated to the most recent `FULL_CTX` only if the rollout ever overflows.


In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

RTG_TARGET   = 20.0
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

hist_state, hist_action, hist_rtg = [], [], []

frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d).astype(np.float32)
    hist_state.append(s_vec)
    hist_rtg.append(np.array([RTG_TARGET], dtype=np.float32))
    if len(hist_action) < len(hist_state):
        hist_action.append(np.zeros(5, dtype=np.float32))   # placeholder for current step

    L = len(hist_state)
    if L > FULL_CTX:
        hist_state  = hist_state[-FULL_CTX:]
        hist_action = hist_action[-FULL_CTX:]
        hist_rtg    = hist_rtg[-FULL_CTX:]
        L = FULL_CTX
    rtg_t    = torch.from_numpy(np.stack(hist_rtg)).unsqueeze(0).to(device)
    state_t  = torch.from_numpy(np.stack(hist_state)).unsqueeze(0).to(device)
    action_t = torch.from_numpy(np.stack(hist_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(rtg_t, state_t, action_t)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards} L={L}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    hist_action[-1] = oh
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')


## 5. Watch the movie


In [ ]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)
